<a href="https://colab.research.google.com/github/hamsamahmoud-cpu/DistilBERT-Fine-tuning-for-Movie-Review-Sentiment-Analysis1/blob/main/03_lstm_and_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download and unzip the GloVe 100-dimensional embeddings
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip -d glove
print("GloVe embeddings downloaded successfully!")

In [ ]:
import torch
import torch.nn as nn

class LSTMSentimentBaseline(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        # The embedding layer turns word IDs into dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # The LSTM processes the sequence
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

        # The classification head outputs Positive (1) or Negative (0)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        # text shape: [batch_size, sequence_length]
        embedded = self.embedding(text)

        # output contains the hidden states for every time step
        # hidden contains the final state
        output, (hidden, cell) = self.lstm(embedded)

        # We only care about the final hidden state to make our prediction
        final_hidden = hidden[-1]
        return self.fc(final_hidden)

print("LSTM Baseline architecture defined.")

In [ ]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from google.colab import drive

# Reconnect to drive if needed
drive.mount('/content/drive')
MODEL_SAVE_PATH = '/content/drive/MyDrive/distilbert-imdb-sentiment/distilbert-finetuned'

print("Loading fine-tuned DistilBERT...")
# We explicitly set output_attentions=True to grab the inner workings!
tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH, output_attentions=True)

In [ ]:
import torch

# 1. Create a sample review
sample_review = "This movie was absolutely brilliant and I loved every second of it!"

# 2. Tokenize the review
inputs = tokenizer(sample_review, return_tensors="pt")

# 3. Pass through the model without calculating gradients
with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract Attention
# outputs.attentions is a tuple of matrices, one for each of the 6 layers.
# We want the last layer (index -1).
attention_last_layer = outputs.attentions[-1]

# The shape is (batch_size, num_heads, sequence_length, sequence_length)
# We will look at the first head [0, 0] and the attention from the [CLS] token (index 0) to all other tokens.
cls_attention = attention_last_layer[0, 0, 0, :]

# 5. Pair the attention weights with their actual tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Attention weights for the [CLS] token:\n")
for token, weight in zip(tokens, cls_attention):
    # Print the token and its formatted percentage of attention
    print(f"{token:15} : {weight.item() * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Filter out the special [CLS] and [SEP] tokens for a cleaner graph
words = []
weights = []
for token, weight in zip(tokens, cls_attention):
    if token not in ['[CLS]', '[SEP]']:
        words.append(token)
        weights.append(weight.item())

# Create the Bar Chart
plt.figure(figsize=(10, 5))
plt.bar(words, weights, color='royalblue')
plt.xlabel('Words in Review', fontsize=12)
plt.ylabel('Attention Weight (Importance)', fontsize=12)
plt.title('What DistilBERT "Looked At" to Make its Decision', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()